In [1]:
# =========================================================
# RPDA8412 RESEARCH PROJECT
# NOTEBOOK 02: DATA PREPROCESSING
#
# Research Period: January 2010 - December 2025
# =========================================================

from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FINAL_DIR = PROJECT_ROOT / "data" / "final"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = pd.Timestamp("2010-01-01")
END_DATE = pd.Timestamp("2025-12-31")

print("=" * 80)
print("RPDA8412 DATA PREPROCESSING")
print("=" * 80)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)

print("\nRaw Files:")

for file in sorted(RAW_DIR.iterdir()):
    print("-", file.name)

RPDA8412 DATA PREPROCESSING
Project root: C:\Users\ibrah\RPDA8412 Research Project
Raw data directory: C:\Users\ibrah\RPDA8412 Research Project\data\raw
Processed data directory: C:\Users\ibrah\RPDA8412 Research Project\data\processed

Raw Files:
- data_gpr_daily_recent_Daily.xls
- data_gpr_export_Monthly.xls
- epu_daily_raw.csv
- epu_monthly_raw.csv
- sp500_daily_raw.csv
- USEPUINDXD_Daily.xlsx
- USEPUINDXD_Monthly.xlsx
- USEPUINDXD_Weekly.xlsx


In [2]:
# =========================================================
# STEP 1: CLEAN S&P 500 DAILY DATA
# =========================================================

print("=" * 60)
print("CLEANING S&P 500 DATA")
print("=" * 60)

# Load original raw dataset
sp500_raw = pd.read_csv(
    RAW_DIR / "sp500_daily_raw.csv"
)

print("Original shape:", sp500_raw.shape)

# Rename date column
sp500 = sp500_raw.rename(
    columns={"Price": "Date"}
).copy()

# Convert dates
# Metadata rows will become NaT
sp500["Date"] = pd.to_datetime(
    sp500["Date"],
    errors="coerce"
)

# Remove rows without valid dates
sp500 = sp500.dropna(
    subset=["Date"]
).copy()

# Standardise column names
sp500 = sp500.rename(
    columns={
        "Adj Close": "Adj_Close",
        "Close": "Close",
        "High": "High",
        "Low": "Low",
        "Open": "Open",
        "Volume": "Volume"
    }
)

# Convert market variables to numeric
numeric_columns = [
    "Adj_Close",
    "Close",
    "High",
    "Low",
    "Open",
    "Volume"
]

for col in numeric_columns:
    sp500[col] = pd.to_numeric(
        sp500[col],
        errors="coerce"
    )

# Restrict research period
sp500 = sp500[
    sp500["Date"].between(
        START_DATE,
        END_DATE
    )
].copy()

# Sort chronologically
sp500 = sp500.sort_values("Date")

# Remove duplicate dates
sp500 = sp500.drop_duplicates(
    subset=["Date"],
    keep="first"
)

# Reset index
sp500 = sp500.reset_index(drop=True)

print("\nCleaned shape:", sp500.shape)

print("\nDate range:")
print(sp500["Date"].min())
print(sp500["Date"].max())

print("\nData types:")
print(sp500.dtypes)

print("\nMissing values:")
print(sp500.isna().sum())

print("\nDuplicate dates:")
print(sp500["Date"].duplicated().sum())

print("\nFirst 5 rows:")
display(sp500.head())

print("\nLast 5 rows:")
display(sp500.tail())

CLEANING S&P 500 DATA
Original shape: (4026, 7)

Cleaned shape: (4024, 7)

Date range:
2010-01-04 00:00:00
2025-12-31 00:00:00

Data types:
Date         datetime64[us]
Adj_Close           float64
Close               float64
High                float64
Low                 float64
Open                float64
Volume                int64
dtype: object

Missing values:
Date         0
Adj_Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64

Duplicate dates:
0

First 5 rows:


C:\Users\ibrah\AppData\Local\Temp\ipykernel_44600\2494700249.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sp500["Date"] = pd.to_datetime(


,Date,Adj_Close,Close,High,Low,Open,Volume
0,2010-01-04,1132.989990,1132.989990,1133.869995,1116.560059,1116.560059,3991400000
1,2010-01-05,1136.520020,1136.520020,1136.630005,1129.660034,1132.660034,2491020000
2,2010-01-06,1137.140015,1137.140015,1139.189941,1133.949951,1135.709961,4972660000
3,2010-01-07,1141.689941,1141.689941,1142.459961,1131.319946,1136.270020,5270680000
4,2010-01-08,1144.979980,1144.979980,1145.390015,1136.219971,1140.520020,4389590000



Last 5 rows:


,Date,Adj_Close,Close,High,Low,Open,Volume
4019,2025-12-24,6932.049805,6932.049805,6937.319824,6904.910156,6904.910156,1798270000
4020,2025-12-26,6929.939941,6929.939941,6945.770020,6921.600098,6936.020020,2586550000
4021,2025-12-29,6905.740234,6905.740234,6920.209961,6888.759766,6903.600098,3541750000
4022,2025-12-30,6896.240234,6896.240234,6913.250000,6893.470215,6900.439941,3309930000
4023,2025-12-31,6845.500000,6845.500000,6901.419922,6844.549805,6898.819824,3261830000


In [3]:
# =========================================================
# STEP 2: VALIDATE AND SAVE CLEANED S&P 500
# =========================================================

print("=" * 60)
print("S&P 500 FINAL VALIDATION")
print("=" * 60)

# Confirm required columns exist
required_columns = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Adj_Close",
    "Volume"
]

assert all(
    col in sp500.columns
    for col in required_columns
), "Missing required columns!"

# Check missing values
assert not sp500[required_columns].isna().any().any(), (
    "Missing values detected!"
)

# Check duplicate dates
assert sp500["Date"].is_unique, (
    "Duplicate dates detected!"
)

# Check chronological order
assert sp500["Date"].is_monotonic_increasing, (
    "Dates are not sorted!"
)

# Check prices are positive
price_columns = [
    "Open",
    "High",
    "Low",
    "Close",
    "Adj_Close"
]

assert (
    sp500[price_columns] > 0
).all().all(), "Invalid prices detected!"

# Save processed dataset
output_file = (
    PROCESSED_DIR / "sp500_daily_clean.csv"
)

sp500.to_csv(
    output_file,
    index=False
)

print("\nALL VALIDATION CHECKS PASSED!")

print("\nSaved to:")
print(output_file)

print("\nFinal observations:", len(sp500))

S&P 500 FINAL VALIDATION

ALL VALIDATION CHECKS PASSED!

Saved to:
C:\Users\ibrah\RPDA8412 Research Project\data\processed\sp500_daily_clean.csv

Final observations: 4024


In [4]:
# =========================================================
# STEP 3: CLEAN GPR DAILY DATA
# =========================================================

print("=" * 60)
print("CLEANING GPR DAILY DATA")
print("=" * 60)

# Load raw dataset
gpr_daily_raw = pd.read_excel(
    RAW_DIR / "data_gpr_daily_recent_Daily.xls"
)

print("Original shape:", gpr_daily_raw.shape)

# Select relevant columns
gpr_daily = gpr_daily_raw[
    [
        "date",
        "GPRD",
        "GPRD_ACT",
        "GPRD_THREAT"
    ]
].copy()

# Standardise column names
gpr_daily = gpr_daily.rename(
    columns={
        "date": "Date",
        "GPRD": "GPR",
        "GPRD_ACT": "GPR_Acts",
        "GPRD_THREAT": "GPR_Threats"
    }
)

# Convert dates
gpr_daily["Date"] = pd.to_datetime(
    gpr_daily["Date"],
    errors="coerce"
)

# Convert GPR values to numeric
numeric_columns = [
    "GPR",
    "GPR_Acts",
    "GPR_Threats"
]

for col in numeric_columns:
    gpr_daily[col] = pd.to_numeric(
        gpr_daily[col],
        errors="coerce"
    )

# Restrict research period
gpr_daily = gpr_daily[
    gpr_daily["Date"].between(
        START_DATE,
        END_DATE
    )
].copy()

# Sort dates
gpr_daily = gpr_daily.sort_values("Date")

# Remove duplicate dates
gpr_daily = gpr_daily.drop_duplicates(
    subset=["Date"],
    keep="first"
)

# Reset index
gpr_daily = gpr_daily.reset_index(drop=True)

print("\nCleaned shape:", gpr_daily.shape)

print("\nDate range:")
print(gpr_daily["Date"].min())
print(gpr_daily["Date"].max())

print("\nMissing values:")
print(gpr_daily.isna().sum())

print("\nDuplicate dates:")
print(gpr_daily["Date"].duplicated().sum())

print("\nFirst 5 rows:")
display(gpr_daily.head())

print("\nLast 5 rows:")
display(gpr_daily.tail())

CLEANING GPR DAILY DATA
Original shape: (15253, 11)

Cleaned shape: (5844, 4)

Date range:
2010-01-01 00:00:00
2025-12-31 00:00:00

Missing values:
Date           0
GPR            0
GPR_Acts       0
GPR_Threats    0
dtype: int64

Duplicate dates:
0

First 5 rows:


,Date,GPR,GPR_Acts,GPR_Threats
0,2010-01-01,200.210129,241.931976,171.525879
1,2010-01-02,110.146950,113.874870,94.982887
2,2010-01-03,91.471298,104.216850,86.927147
3,2010-01-04,177.117065,190.888504,214.946838
4,2010-01-05,122.525620,148.323151,123.716156



Last 5 rows:


,Date,GPR,GPR_Acts,GPR_Threats
5839,2025-12-27,223.724625,187.062424,240.043900
5840,2025-12-28,79.820908,79.575211,99.560371
5841,2025-12-29,62.946659,47.064674,65.427643
5842,2025-12-30,218.892136,160.026810,266.956329
5843,2025-12-31,161.760208,140.228073,210.535324


In [5]:
# =========================================================
# STEP 4: VALIDATE AND SAVE GPR DAILY
# =========================================================

print("=" * 60)
print("GPR DAILY VALIDATION")
print("=" * 60)

assert len(gpr_daily) == 5844, (
    "Unexpected number of daily observations!"
)

assert gpr_daily["Date"].is_unique, (
    "Duplicate dates detected!"
)

assert gpr_daily["Date"].is_monotonic_increasing, (
    "Dates are not sorted!"
)

assert gpr_daily["GPR"].notna().all(), (
    "Missing GPR values detected!"
)

assert (gpr_daily["GPR"] >= 0).all(), (
    "Negative GPR values detected!"
)

output_file = (
    PROCESSED_DIR / "gpr_daily_clean.csv"
)

gpr_daily.to_csv(
    output_file,
    index=False
)

print("\nALL GPR DAILY VALIDATION CHECKS PASSED!")

print("\nSaved to:")
print(output_file)

print("\nObservations:", len(gpr_daily))

GPR DAILY VALIDATION

ALL GPR DAILY VALIDATION CHECKS PASSED!

Saved to:
C:\Users\ibrah\RPDA8412 Research Project\data\processed\gpr_daily_clean.csv

Observations: 5844


In [6]:
# =========================================================
# STEP 5: CLEAN GPR MONTHLY DATA
# =========================================================

print("=" * 60)
print("CLEANING GPR MONTHLY DATA")
print("=" * 60)

gpr_monthly_raw = pd.read_excel(
    RAW_DIR / "data_gpr_export_Monthly.xls"
)

print("Original shape:", gpr_monthly_raw.shape)

# Select relevant variables
gpr_monthly = gpr_monthly_raw[
    [
        "month",
        "GPR",
        "GPRT",
        "GPRA"
    ]
].copy()

# Rename columns
gpr_monthly = gpr_monthly.rename(
    columns={
        "month": "Date",
        "GPR": "GPR",
        "GPRT": "GPR_Threats",
        "GPRA": "GPR_Acts"
    }
)

# Convert dates
gpr_monthly["Date"] = pd.to_datetime(
    gpr_monthly["Date"],
    errors="coerce"
)

# Convert numeric variables
for col in [
    "GPR",
    "GPR_Threats",
    "GPR_Acts"
]:
    gpr_monthly[col] = pd.to_numeric(
        gpr_monthly[col],
        errors="coerce"
    )

# Filter research period
gpr_monthly = gpr_monthly[
    gpr_monthly["Date"].between(
        START_DATE,
        END_DATE
    )
].copy()

# Sort dates
gpr_monthly = gpr_monthly.sort_values("Date")

# Remove duplicate dates
gpr_monthly = gpr_monthly.drop_duplicates(
    subset=["Date"],
    keep="first"
)

# Reset index
gpr_monthly = gpr_monthly.reset_index(drop=True)

print("\nCleaned shape:", gpr_monthly.shape)

print("\nDate range:")
print(gpr_monthly["Date"].min())
print(gpr_monthly["Date"].max())

print("\nMissing values:")
print(gpr_monthly.isna().sum())

print("\nDuplicate dates:")
print(gpr_monthly["Date"].duplicated().sum())

print("\nFirst 5 rows:")
display(gpr_monthly.head())

print("\nLast 5 rows:")
display(gpr_monthly.tail())

CLEANING GPR MONTHLY DATA
Original shape: (1521, 115)

Cleaned shape: (192, 4)

Date range:
2010-01-01 00:00:00
2025-12-01 00:00:00

Missing values:
Date           0
GPR            0
GPR_Threats    0
GPR_Acts       0
dtype: int64

Duplicate dates:
0

First 5 rows:


,Date,GPR,GPR_Threats,GPR_Acts
0,2010-01-01,91.581024,84.972992,100.410942
1,2010-02-01,80.725357,78.846275,80.711739
2,2010-03-01,74.116943,73.496910,70.039345
3,2010-04-01,88.761581,92.056915,81.268402
4,2010-05-01,88.958710,87.494110,85.857391



Last 5 rows:


,Date,GPR,GPR_Threats,GPR_Acts
187,2025-08-01,138.988464,160.415192,121.731056
188,2025-09-01,124.127380,139.635590,113.484726
189,2025-10-01,154.314072,167.662247,151.558167
190,2025-11-01,104.155319,117.667854,89.126900
191,2025-12-01,132.486221,146.358932,123.534622


In [7]:
# =========================================================
# STEP 6: VALIDATE AND SAVE GPR MONTHLY
# =========================================================

print("=" * 60)
print("GPR MONTHLY VALIDATION")
print("=" * 60)

assert len(gpr_monthly) == 192, (
    "Unexpected number of monthly observations!"
)

assert gpr_monthly["Date"].is_unique, (
    "Duplicate dates detected!"
)

assert gpr_monthly["Date"].is_monotonic_increasing, (
    "Dates are not sorted!"
)

assert gpr_monthly["GPR"].notna().all(), (
    "Missing GPR values detected!"
)

assert (gpr_monthly["GPR"] >= 0).all(), (
    "Negative GPR values detected!"
)

output_file = (
    PROCESSED_DIR / "gpr_monthly_clean.csv"
)

gpr_monthly.to_csv(
    output_file,
    index=False
)

print("\nALL GPR MONTHLY VALIDATION CHECKS PASSED!")

print("\nSaved to:")
print(output_file)

print("\nObservations:", len(gpr_monthly))

GPR MONTHLY VALIDATION

ALL GPR MONTHLY VALIDATION CHECKS PASSED!

Saved to:
C:\Users\ibrah\RPDA8412 Research Project\data\processed\gpr_monthly_clean.csv

Observations: 192


In [8]:
# =========================================================
# STEP 7: CLEAN EPU DAILY DATA
# =========================================================

print("=" * 60)
print("CLEANING EPU DAILY DATA")
print("=" * 60)

# Load raw daily EPU data
epu_daily_raw = pd.read_csv(
    RAW_DIR / "epu_daily_raw.csv"
)

print("Original shape:", epu_daily_raw.shape)

# Select required columns
epu_daily = epu_daily_raw[
    ["observation_date", "USEPUINDXD"]
].copy()

# Standardise column names
epu_daily = epu_daily.rename(
    columns={
        "observation_date": "Date",
        "USEPUINDXD": "EPU"
    }
)

# Convert date column
epu_daily["Date"] = pd.to_datetime(
    epu_daily["Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Convert EPU values to numeric
epu_daily["EPU"] = pd.to_numeric(
    epu_daily["EPU"],
    errors="coerce"
)

# Restrict to research period
epu_daily = epu_daily[
    epu_daily["Date"].between(
        START_DATE,
        END_DATE
    )
].copy()

# Sort chronologically
epu_daily = epu_daily.sort_values("Date")

# Remove duplicate dates
epu_daily = epu_daily.drop_duplicates(
    subset=["Date"],
    keep="first"
)

# Reset index
epu_daily = epu_daily.reset_index(drop=True)

print("\nCleaned shape:", epu_daily.shape)

print("\nDate range:")
print(epu_daily["Date"].min())
print(epu_daily["Date"].max())

print("\nData types:")
print(epu_daily.dtypes)

print("\nMissing values:")
print(epu_daily.isna().sum())

print("\nDuplicate dates:")
print(epu_daily["Date"].duplicated().sum())

print("\nFirst 5 rows:")
display(epu_daily.head())

print("\nLast 5 rows:")
display(epu_daily.tail())

CLEANING EPU DAILY DATA
Original shape: (15254, 2)

Cleaned shape: (5844, 2)

Date range:
2010-01-01 00:00:00
2025-12-31 00:00:00

Data types:
Date    datetime64[us]
EPU            float64
dtype: object

Missing values:
Date    0
EPU     0
dtype: int64

Duplicate dates:
0

First 5 rows:


,Date,EPU
0,2010-01-01,174.55
1,2010-01-02,119.52
2,2010-01-03,216.95
3,2010-01-04,150.35
4,2010-01-05,151.16



Last 5 rows:


,Date,EPU
5839,2025-12-27,281.48
5840,2025-12-28,462.89
5841,2025-12-29,445.72
5842,2025-12-30,305.80
5843,2025-12-31,307.63


In [9]:
# =========================================================
# STEP 8: VALIDATE AND SAVE EPU DAILY
# =========================================================

print("=" * 60)
print("EPU DAILY VALIDATION")
print("=" * 60)

assert len(epu_daily) == 5844, (
    "Unexpected number of daily observations!"
)

assert epu_daily["Date"].is_unique, (
    "Duplicate dates detected!"
)

assert epu_daily["Date"].is_monotonic_increasing, (
    "Dates are not sorted!"
)

assert epu_daily["EPU"].notna().all(), (
    "Missing EPU values detected!"
)

assert (epu_daily["EPU"] >= 0).all(), (
    "Negative EPU values detected!"
)

# Verify complete calendar-day coverage
expected_dates = pd.date_range(
    START_DATE,
    END_DATE,
    freq="D"
)

assert len(
    expected_dates.difference(epu_daily["Date"])
) == 0, "Missing calendar dates detected!"

# Save processed dataset
output_file = PROCESSED_DIR / "epu_daily_clean.csv"

epu_daily.to_csv(
    output_file,
    index=False
)

print("\nALL EPU DAILY VALIDATION CHECKS PASSED!")
print("Saved to:", output_file)
print("Observations:", len(epu_daily))

EPU DAILY VALIDATION

ALL EPU DAILY VALIDATION CHECKS PASSED!
Saved to: C:\Users\ibrah\RPDA8412 Research Project\data\processed\epu_daily_clean.csv
Observations: 5844


In [10]:
# =========================================================
# STEP 9: CLEAN EPU MONTHLY DATA
# =========================================================

print("=" * 60)
print("CLEANING EPU MONTHLY DATA")
print("=" * 60)

# Load raw monthly EPU data
epu_monthly_raw = pd.read_csv(
    RAW_DIR / "epu_monthly_raw.csv"
)

print("Original shape:", epu_monthly_raw.shape)

# Select required columns
epu_monthly = epu_monthly_raw[
    ["observation_date", "USEPUINDXM"]
].copy()

# Standardise column names
epu_monthly = epu_monthly.rename(
    columns={
        "observation_date": "Date",
        "USEPUINDXM": "EPU"
    }
)

# Convert dates
epu_monthly["Date"] = pd.to_datetime(
    epu_monthly["Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Convert numeric values
epu_monthly["EPU"] = pd.to_numeric(
    epu_monthly["EPU"],
    errors="coerce"
)

# Restrict research period
epu_monthly = epu_monthly[
    epu_monthly["Date"].between(
        START_DATE,
        END_DATE
    )
].copy()

# Sort chronologically
epu_monthly = epu_monthly.sort_values("Date")

# Remove duplicate dates
epu_monthly = epu_monthly.drop_duplicates(
    subset=["Date"],
    keep="first"
)

# Reset index
epu_monthly = epu_monthly.reset_index(drop=True)

print("\nCleaned shape:", epu_monthly.shape)

print("\nDate range:")
print(epu_monthly["Date"].min())
print(epu_monthly["Date"].max())

print("\nMissing values:")
print(epu_monthly.isna().sum())

print("\nDuplicate dates:")
print(epu_monthly["Date"].duplicated().sum())

print("\nFirst 5 rows:")
display(epu_monthly.head())

print("\nLast 5 rows:")
display(epu_monthly.tail())

CLEANING EPU MONTHLY DATA
Original shape: (501, 2)

Cleaned shape: (192, 2)

Date range:
2010-01-01 00:00:00
2025-12-01 00:00:00

Missing values:
Date    0
EPU     0
dtype: int64

Duplicate dates:
0

First 5 rows:


,Date,EPU
0,2010-01-01,149.35928
1,2010-02-01,132.79865
2,2010-03-01,137.47003
3,2010-04-01,137.07740
4,2010-05-01,151.10342



Last 5 rows:


,Date,EPU
187,2025-08-01,254.12744
188,2025-09-01,224.23756
189,2025-10-01,217.50845
190,2025-11-01,218.76603
191,2025-12-01,205.27646


In [11]:
# =========================================================
# STEP 10: VALIDATE AND SAVE EPU MONTHLY
# =========================================================

print("=" * 60)
print("EPU MONTHLY VALIDATION")
print("=" * 60)

assert len(epu_monthly) == 192, (
    "Unexpected number of monthly observations!"
)

assert epu_monthly["Date"].is_unique, (
    "Duplicate dates detected!"
)

assert epu_monthly["Date"].is_monotonic_increasing, (
    "Dates are not sorted!"
)

assert epu_monthly["EPU"].notna().all(), (
    "Missing EPU values detected!"
)

assert (epu_monthly["EPU"] >= 0).all(), (
    "Negative EPU values detected!"
)

# Check complete monthly coverage
expected_months = pd.date_range(
    "2010-01-01",
    "2025-12-01",
    freq="MS"
)

assert len(
    expected_months.difference(epu_monthly["Date"])
) == 0, "Missing monthly observations detected!"

# Save processed dataset
output_file = PROCESSED_DIR / "epu_monthly_clean.csv"

epu_monthly.to_csv(
    output_file,
    index=False
)

print("\nALL EPU MONTHLY VALIDATION CHECKS PASSED!")
print("Saved to:", output_file)
print("Observations:", len(epu_monthly))

EPU MONTHLY VALIDATION

ALL EPU MONTHLY VALIDATION CHECKS PASSED!
Saved to: C:\Users\ibrah\RPDA8412 Research Project\data\processed\epu_monthly_clean.csv
Observations: 192
